In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 1. 定义数据 (SGP 和 Laplacian)
method_list = ["SD-1.5", "Diffusion-DPO", "Diffusion-KTO", "InPO", "Ours", "Diffusion-DPO+Ours"]

laplacian_raw = {
    "SD-1.5": 1832.336, "Diffusion-DPO": 1552.312,
    "Diffusion-KTO": 1580.512, "InPO": 1255.905,
    "Ours": 2353.761, "Diffusion-DPO+Ours": 2345.722
}

sgp_raw = {
    "SD-1.5": -0.041, "Diffusion-DPO": -0.075,
    "Diffusion-KTO": -0.221, "InPO": -0.253,
    "Ours": -0.009, "Diffusion-DPO+Ours": -0.011,
}

# 2. 归一化处理
def normalize(val, min_val, max_val):
    return (val - min_val) / (max_val - min_val)

laps = list(laplacian_raw.values()); sgps = list(sgp_raw.values())
lap_min, lap_max = min(laps), max(laps)
sgp_min, sgp_max = min(sgps), max(sgps)

# 3. 绘图设置 (严格同步你提供的样式)
plt.rcParams.update({
    "axes.labelsize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10,
})

fig, ax = plt.subplots(figsize=(4, 4), dpi=300)

# 严格对齐的样式字典
base_size = 280
style_mapping = {
    "SD-1.5":        {"mk": "o", "c": "#A4C6E2", "s": base_size, "z": 3},
    "Diffusion-DPO": {"mk": "D", "c": "#AFD498", "s": 90/160*base_size, "z": 3}, # FlowGRPO
    "Diffusion-KTO": {"mk": "P", "c": "#AE9DC7", "s": base_size,  "z": 3}, # 仿照 GRPO-Guard
    "InPO":          {"mk": "X", "c": "#e7dbd3", "s": base_size, "z": 3}, # AFPO (HPDv3)
    "Ours":          {"mk": "s", "c": "#E4A39D", "s": base_size*0.75, "z": 11}, # AFPO (HPDv3)
    "Diffusion-DPO+Ours": {"mk": "h", "c": "#D86983", "s": base_size, "z": 11} # AFPO (HPDv3)
}

# 绘制点
for m in method_list:
    x = normalize(sgp_raw[m], sgp_min, sgp_max)
    y = normalize(laplacian_raw[m], lap_min, lap_max)
    st = style_mapping[m]
    
    ax.scatter(
        x, y,
        s=st["s"], marker=st["mk"], facecolors=st["c"], edgecolors="black",
        linewidths=1.0, 
        zorder=st["z"], label=m, 
    )

# 4. 标签布局 (微调偏移量以防重叠)
label_offsets = {
    "SD-1.5": (0, 15),
    "Diffusion-DPO": (-10, 15),
    "Diffusion-KTO": (5, 10),
    "Ours": (-20, 12),
    "Diffusion-DPO+Ours": (-20, -22),
    "InPO": (10, 0),
}

fontsize_label = 11
for m in method_list:
    x = normalize(sgp_raw[m], sgp_min, sgp_max)
    y = normalize(laplacian_raw[m], lap_min, lap_max)
    dx, dy = label_offsets.get(m, (5, 5))
    
    ha_align = 'center' if m == "Diffusion-DPO+Ours" else 'left'
    # 1. 优先处理带加号的复杂情况
    if m == "Diffusion-DPO+Ours":
        display_name = r"Diffusion-DPO" + "\n" + r"$\mathbf{+Ours}$"
    # 2. 再处理单独的 Ours
    elif "Ours" in m:
        display_name = m.replace("Ours", r"$\mathbf{Ours}$")
    else:
        display_name = m
    ax.annotate(
        display_name, (x, y), xytext=(dx, dy), textcoords="offset points",
        fontsize=fontsize_label, color="#333333", fontweight="normal",
        ha=ha_align, va='center', zorder=20
    )

# 坐标轴设置
ax.set_xlabel(r"Realism$\uparrow$")
ax.set_ylabel(r"Texture Detail$\uparrow$")
ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.3, zorder=0)
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# 范围刻度
ax.set_xlim(-0.1, 1.1); ax.set_xticks(np.arange(0.0, 1.1, 0.2))
ax.set_ylim(-0.1, 1.1); ax.set_yticks(np.arange(0.0, 1.1, 0.2))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.1f}"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.1f}"))

# 图例 (严格对齐)
legend = ax.legend(
    loc="upper left", frameon=True, fancybox=True, edgecolor="gray",
    framealpha=0.3, borderpad=0.3, labelspacing=0.6,
    handletextpad=0.3, borderaxespad=0.8
)

# 修正图例 Marker 大小
for handle in getattr(legend, 'legend_handles', []):
    lbl = handle.get_label()
    if lbl == "Diffusion-DPO+Ours":
        handle.set_sizes([120*1.25])
    elif lbl == "Ours":
        handle.set_sizes([80/100*120])
    elif lbl == "Diffusion-DPO":
        handle.set_sizes([60/100*120])
    else:
        handle.set_sizes([100.0/100*120])

fig.tight_layout()
fig.savefig("detail-vs-realism.pdf", dpi=300, bbox_inches="tight")
plt.show()